[Optiver - Trading at the Close](https://www.kaggle.com/competitions/optiver-trading-at-the-close)

[PyTorch DNN](https://www.kaggle.com/code/xquantum/pytorch-dnn)

# Overview
This notebook contains a pipeline for a __PyTorch DNN__ to __predict stock movements__ for the __Optiver Trading Challenge__.

- Customizable Neural Network
- Preprocessing and Normalization of Input Data
- Feature Engineering
- Decaying Learning Rate and Early Stopping
- Hardware Acceleration

## Possible Improvements
- [PyTorch Profiler](https://docs.pytorch.org/tutorials/recipes/recipes/profiler_recipe.html) for Bottlenecks
- Hyperparameter Tuning with [RayTune](https://docs.ray.io/en/latest/tune/index.html) (lr, layers, batchsize)
- Flag filled NaN for near_price and far_price
- Regularisation methods for deeper networks (batchnorm)there is a huge improvement in how you deal with the unknown clips :-D

# DeepL 번역
이 노트북에는 __Optiver Trading Challenge__를 위해 __주식 시세 변동__을 예측하는 __PyTorch DNN__ 파이프라인이 포함되어 있습니다.
- 사용자 정의 가능한 신경망
- 입력 데이터의 전처리 및 정규화
- 특징 공학
- 학습률 감쇠 및 조기 종료
- 하드웨어 가속
## 개선 방안
- 병목 현상 분석을 위한 [PyTorch Profiler](https://docs.pytorch.org/tutorials/recipes/recipes/profiler_recipe.html)
- [RayTune](https://docs.ray.io/en/latest/tune/index.html)을 이용한 하이퍼파라미터 튜닝 (학습률, 레이어, 배치 크기)
- near_price 및 far_price에 대해 NaN으로 채우기
- 더 깊은 네트워크를 위한 정규화 방법 (batchnorm) 미지 클립을 처리하는 방식에서 엄청난 개선이 있습니다 :-D

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
plt.tight_layout()
plt.rcParams["figure.figsize"] = (6, 3)
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None);
# pd.reset_option('display.max_columns');

<Figure size 640x480 with 0 Axes>

In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from fastai.tabular.all import *

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


# Data

In [4]:
df_raw = pd.read_csv("./input/007_optiver-trading-at-the-close/train.csv")
df_raw.isna().sum(axis=0) / len(df_raw)
display(df_raw)

,stock_id,date_id,seconds_in_bucket,imbalance_size,imbalance_buy_sell_flag,reference_price,matched_size,far_price,near_price,bid_price,bid_size,ask_price,ask_size,wap,target,time_id,row_id
0,0,0,0,3180602.69,1,0.999812,13380276.64,NaN,NaN,0.999812,60651.50,1.000026,8493.03,1.000000,-3.029704,0,0_0_0
1,1,0,0,166603.91,-1,0.999896,1642214.25,NaN,NaN,0.999896,3233.04,1.000660,20605.09,1.000000,-5.519986,0,0_0_1
2,2,0,0,302879.87,-1,0.999561,1819368.03,NaN,NaN,0.999403,37956.00,1.000298,18995.00,1.000000,-8.389950,0,0_0_2
3,3,0,0,11917682.27,-1,1.000171,18389745.62,NaN,NaN,0.999999,2324.90,1.000214,479032.40,1.000000,-4.010200,0,0_0_3
4,4,0,0,447549.96,-1,0.999532,17860614.95,NaN,NaN,0.999394,16485.54,1.000016,434.10,1.000000,-7.349849,0,0_0_4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5237975,195,480,540,2440722.89,-1,1.000317,28280361.74,0.999734,0.999734,1.000317,32257.04,1.000434,319862.40,1.000328,2.310276,26454,480_540_195
5237976,196,480,540,349510.47,-1,1.000643,9187699.11,1.000129,1.000386,1.000643,205108.40,1.000900,93393.07,1.000819,-8.220077,26454,480_540_196
5237977,197,480,540,0.00,0,0.995789,12725436.10,0.995789,0.995789,0.995789,16790.66,0.995883,180038.32,0.995797,1.169443,26454,480_540_197
5237978,198,480,540,1000898.84,1,0.999210,94773271.05,0.999210,0.999210,0.998970,125631.72,0.999210,669893.00,0.999008,-1.540184,26454,480_540_198


In [5]:
df_raw.describe()

,stock_id,date_id,seconds_in_bucket,imbalance_size,imbalance_buy_sell_flag,reference_price,matched_size,far_price,near_price,bid_price,bid_size,ask_price,ask_size,wap,target,time_id
count,5.237980e+06,5.237980e+06,5.237980e+06,5.237760e+06,5.237980e+06,5.237760e+06,5.237760e+06,2.343638e+06,2.380800e+06,5.237760e+06,5.237980e+06,5.237760e+06,5.237980e+06,5.237760e+06,5.237892e+06,5.237980e+06
mean,9.928856e+01,2.415100e+02,2.700000e+02,5.715293e+06,-1.189619e-02,9.999955e-01,4.510025e+07,1.001713e+00,9.996601e-01,9.997263e-01,5.181359e+04,1.000264e+00,5.357568e+04,9.999920e-01,-4.756125e-02,1.331005e+04
std,5.787176e+01,1.385319e+02,1.587451e+02,2.051591e+07,8.853374e-01,2.532497e-03,1.398413e+08,7.214705e-01,1.216920e-02,2.499345e-03,1.114214e+05,2.510042e-03,1.293554e+05,2.497509e-03,9.452860e+00,7.619271e+03
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-1.000000e+00,9.352850e-01,4.316610e+03,7.700000e-05,7.869880e-01,9.349150e-01,0.000000e+00,9.398270e-01,0.000000e+00,9.380080e-01,-3.852898e+02,0.000000e+00
25%,4.900000e+01,1.220000e+02,1.300000e+02,8.453415e+04,-1.000000e+00,9.987630e-01,5.279575e+06,9.963320e-01,9.971000e-01,9.985290e-01,7.374720e+03,9.990290e-01,7.823700e+03,9.987810e-01,-4.559755e+00,6.729000e+03
50%,9.900000e+01,2.420000e+02,2.700000e+02,1.113604e+06,0.000000e+00,9.999670e-01,1.288264e+07,9.998830e-01,9.998890e-01,9.997280e-01,2.196900e+04,1.000207e+00,2.301792e+04,9.999970e-01,-6.020069e-02,1.334500e+04
75%,1.490000e+02,3.610000e+02,4.100000e+02,4.190951e+06,1.000000e+00,1.001174e+00,3.270013e+07,1.003318e+00,1.002590e+00,1.000905e+00,5.583168e+04,1.001414e+00,5.787841e+04,1.001149e+00,4.409552e+00,1.990700e+04
max,1.990000e+02,4.800000e+02,5.400000e+02,2.982028e+09,1.000000e+00,1.077488e+00,7.713682e+09,4.379531e+02,1.309732e+00,1.077488e+00,3.028784e+07,1.077836e+00,5.440500e+07,1.077675e+00,4.460704e+02,2.645400e+04


In [6]:
def add_historic_features(df, cols, shifts=3, add_first=True):
    for col in cols:
        grouped_vals = df[["stock_id", "date_id", col]].groupby(["stock_id", "date_id"])
        fill_value = df[col].mean()

        for shift in np.arange(shifts):
            df[col + "_shift" + str(shift+1)] = grouped_vals.shift(shift+1).fillna(fill_value)
        
        if add_first:
            df = df.merge(grouped_vals.first().reset_index(), on=["date_id", "stock_id"], suffixes=["", "_first"])
            return df

In [7]:
def fillmean(df, cols):
    for col in cols:
        mean_val = df[col].mean()
        df[col] = df[col].fillna(mean_val)
    return df

In [8]:
def add_info_columns(raw_df: pd.DataFrame):
    df = raw_df.copy()
    df[["reference_price", "far_price", "near_price", "bid_price", "ask_price", "wap"]] = df[["reference_price", "far_price", "near_price", "bid_price", "ask_price", "wap"]].fillna(1, axis=0)
    df = fillmean(df, ["imbalance_size", "matched_size"])

    df["imbalance_ratio"] = df["imbalance_size"] / (df["matched_size"] + 1.0e-8)
    df["imbalance"] = df["imbalance_size"] * df["imbalance_buy_sell_flag"]
    
    df["ordersize_imbalance"] = (df["bid_size"] - df["ask_size"]) / ((df["bid_size"] + df["ask_size"]) + 1.0e-8)
    df["matching_imbalance"] = (df["imbalance_size"] - df["matched_size"]) / ((df["imbalance_size"] + df["matched_size"] + 1.0e-8))

    df = add_historic_features(df, ["imbalance", "imbalance_ratio", "reference_price", "wrap_classmatched_size", "far_price", "near_price"], shifts=6, add_first=True)

    return df

In [9]:
df = add_info_columns(df_raw)
nullsum = df.isna().sum(axis=0)
print(nullsum[nullsum != 0])
df.dropna(inplace=True)
display(df);

target    88
dtype: int64


,stock_id,date_id,seconds_in_bucket,imbalance_size,imbalance_buy_sell_flag,reference_price,matched_size,far_price,near_price,bid_price,bid_size,ask_price,ask_size,wap,target,time_id,row_id,imbalance_ratio,imbalance,ordersize_imbalance,matching_imbalance,imbalance_shift1,imbalance_shift2,imbalance_shift3,imbalance_shift4,imbalance_shift5,imbalance_shift6,imbalance_first
0,0,0,0,3180602.69,1,0.999812,13380276.64,1.000000,1.000000,0.999812,60651.50,1.000026,8493.03,1.000000,-3.029704,0,0_0_0,0.237708,3180602.69,0.754340,-0.615890,-2.524997e+05,-2.524997e+05,-2.524997e+05,-2.524997e+05,-2.524997e+05,-2.524997e+05,3180602.69
1,1,0,0,166603.91,-1,0.999896,1642214.25,1.000000,1.000000,0.999896,3233.04,1.000660,20605.09,1.000000,-5.519986,0,0_0_1,0.101451,-166603.91,-0.728751,-0.815787,-2.524997e+05,-2.524997e+05,-2.524997e+05,-2.524997e+05,-2.524997e+05,-2.524997e+05,-166603.91
2,2,0,0,302879.87,-1,0.999561,1819368.03,1.000000,1.000000,0.999403,37956.00,1.000298,18995.00,1.000000,-8.389950,0,0_0_2,0.166475,-302879.87,0.332935,-0.714567,-2.524997e+05,-2.524997e+05,-2.524997e+05,-2.524997e+05,-2.524997e+05,-2.524997e+05,-302879.87
3,3,0,0,11917682.27,-1,1.000171,18389745.62,1.000000,1.000000,0.999999,2324.90,1.000214,479032.40,1.000000,-4.010200,0,0_0_3,0.648061,-11917682.27,-0.990340,-0.213547,-2.524997e+05,-2.524997e+05,-2.524997e+05,-2.524997e+05,-2.524997e+05,-2.524997e+05,-11917682.27
4,4,0,0,447549.96,-1,0.999532,17860614.95,1.000000,1.000000,0.999394,16485.54,1.000016,434.10,1.000000,-7.349849,0,0_0_4,0.025058,-447549.96,0.948687,-0.951109,-2.524997e+05,-2.524997e+05,-2.524997e+05,-2.524997e+05,-2.524997e+05,-2.524997e+05,-447549.96
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5237975,195,480,540,2440722.89,-1,1.000317,28280361.74,0.999734,0.999734,1.000317,32257.04,1.000434,319862.40,1.000328,2.310276,26454,480_540_195,0.086305,-2440722.89,-0.816784,-0.841104,-2.440723e+06,-2.456160e+06,-2.539436e+06,-1.954705e+06,-1.971686e+06,-3.202296e+06,-1580093.42
5237976,196,480,540,349510.47,-1,1.000643,9187699.11,1.000129,1.000386,1.000643,205108.40,1.000900,93393.07,1.000819,-8.220077,26454,480_540_196,0.038041,-349510.47,0.374254,-0.926706,-3.495105e+05,-3.495105e+05,-3.495105e+05,-3.495494e+05,-8.245740e+03,-3.740921e+05,-4198092.93
5237977,197,480,540,0.00,0,0.995789,12725436.10,0.995789,0.995789,0.995789,16790.66,0.995883,180038.32,0.995797,1.169443,26454,480_540_197,0.000000,0.00,-0.829388,-1.000000,-9.286723e+05,-9.779767e+05,-1.008605e+06,-1.084269e+06,-1.084269e+06,-1.109455e+06,-1218735.68
5237978,198,480,540,1000898.84,1,0.999210,94773271.05,0.999210,0.999210,0.998970,125631.72,0.999210,669893.00,0.999008,-1.540184,26454,480_540_198,0.010561,1000898.84,-0.684154,-0.979099,1.000899e+06,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,4377473.60


In [10]:
x_cols = [c for c in df.columns if c not in ["row_id", "time_id", "date_id", "target"]]
y_cols = ["target"]

In [11]:
means = df[x_cols].mean(0)
stds = df[y_cols].std(0)

In [12]:
def normalize_features(x):
    return (x - means) / (stds + 1e-8)

In [13]:
def get_xy(df):
    x = df[x_cols]
    x = normalize_features(x)

    y = df[y_cols]

    return x.values, y.values

In [14]:
def get_dataloaders(df, batch_size=512):
    (x, y) = get_xy(df)

    x_tensor = torch.Tensor(x).to(device)
    y_tensor = torch.Tensor(y).to(device)

    full_dataset = TensorDataset(x_tensor, y_tensor)
    train_dataset, test_dataset = random_split(full_dataset, [0.8, 0.2])

    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    return (train_dataset, test_dataset)

In [15]:
train_dataloader, test_dataloader = get_dataloaders(df)

# Model

In [16]:
layers = [512, 256, 128, 64]

# 아래 코드를 과거 형식으로 풀어서 만든 것
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(len(x_cols), 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc5 = nn.Linear(64, 1)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):

        x = self.fc1(x)
        x = self.relu(x)

        x = self.dropout(x)
        x = self.fc2(x)
        x = self.relu(x)

        x = self.dropout(x)
        x = self.fc3(x)
        x = self.relu(x)

        x = self.dropout(x)
        x = self.fc4(x)
        x = self.relu(x)

        x = self.fc5(x)

        return x

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.relu_stack = nn.Sequential(
            nn.Linear(len(x_cols), layers[0]),
            nn.ReLU()
        )

        for i in range(len(layers) - 1):
            self.relu_stack.append(nn.Dropout(0.25))
            self.relu_stack.append(nn.Linear(layers[i], layers[i+1]))
            self.relu_stack.append(nn.ReLU())
        self.relu_stack.append(nn.Linear(layers[-1], 1))
    
    def forward(self, x):
        output = self.relu_stack(x)
        return output

def init_weights(m):
    if isinstance(m, nn.Linear):
        torch.nn.init.kaiming_normal_(m.weight)
        m.bias.data.fill_(0.01)
        if m.out_features == 1:
            torch.nn.init.xavier_normal_(m.weight)

In [17]:
def train_loop(dataloader, model, loss_fn, optimizer, shortcut=0):
    size = len(dataloader.dataset)
    model.train()
    num_batches = len(dataloader)

    train_loss = 0

    for batch, (X, y) in enumerate(dataloader):
        pred = model(X)

        loss = loss_fn(pred, y)
        train_loss += loss

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if pred.std() < 0.000001:
            print("WARNING: std() is zero, stopping")
            break
        
        if shortcut > 0 and batch == shortcut:
            return train_loss.detach().cpu().numpy() / shortcut
    return train_loss.detach().cput().numpy() / num_batches

def test_loop(dataloader, model, loss_fn):
    model.eval()
    # size = len(dataloader.dataset)  # 사용하는 데가 없어 주석 처리함
    num_batches = len(dataloader)
    test_loss = 0
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).detach().cpu().numpy()

        scheduler.step(test_loss)   # 나중에 전역에서 선언하여 사용함

    return test_loss / num_batches

def predict(X, model):
    model.eval()
    with torch.no_grad():
        pred = model(X)
    return pred.detach().cpu().numpy.flatten()

In [18]:
model = NeuralNetwork().to(device)
print(f"NUmber of parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")
model.apply(init_weights)

NUmber of parameters: 185345


NeuralNetwork(
  (relu_stack): Sequential(
    (0): Linear(in_features=24, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.25, inplace=False)
    (3): Linear(in_features=512, out_features=256, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.25, inplace=False)
    (6): Linear(in_features=256, out_features=128, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.25, inplace=False)
    (9): Linear(in_features=128, out_features=64, bias=True)
    (10): ReLU()
    (11): Linear(in_features=64, out_features=1, bias=True)
  )
)

# Training

https://www.kaggle.com/code/xquantum/pytorch-dnn